# Falsify the finite claims

This lesson attacks the two places where a superficially plausible check can pass: an incomplete G1 path family and cancellation of signed defects.

![Finite defect diagnostics](../public_export/figure_lggr_phase_boundary.png)

In [ ]:
from epk_nb import import_epk, widgets_available
import matplotlib.pyplot as plt

epk = import_epk()
HAS_WIDGETS = widgets_available()

## Falsifier 1: test the converse of G1

If the global-path list is empty, the sentence ‘every observed global path splits’ is vacuously true. It is still insufficient when compatible left and right segments exist. The bijection requires their concatenation to occur exactly once.

In [ ]:
a, b, c = map(epk.State, ['a', 'b', 'c'])
left = epk.TrajectorySegment(a, b, ('left',))
right = epk.TrajectorySegment(b, c, ('right',))
boundary = epk.BoundarySlice({b})
valid, violations = boundary.validate_unique_decomposition(
    [], left_segments=[left], right_segments=[right]
)
print({'G1': valid, 'violations': violations})
assert not valid
assert any('Missing Cartesian product trajectory' in message for message in violations)

## Falsifier 2: zero net residual is not pointwise agreement

One negative and one positive defect sum to zero. The absolute defect rate still records both. If the evaluator and weight both fail to factor, retain all four terms:

$$E_gw_g=E_1E_2w_1w_2+\Delta_Ew_1w_2+E_1E_2\Delta_w+\Delta_E\Delta_w.$$

In [ ]:
sample = [
    epk.EvaluatorVerdictTriple(1, 1, 0),
    epk.EvaluatorVerdictTriple(0, 1, 1),
]
report = epk.EPKMetrics.compute(sample)
expanded = epk.EvaluatedPathKernel.four_term_contribution(1, 1, 2.0, 3.0, -1, 5.0)
print({'signed sum': report.delta_sum, 'EGD': report.egd, 'four-term value': expanded})
assert report.delta_sum == 0 and report.egd == 1.0
assert expanded == (1 * 1 - 1) * (2.0 * 3.0 + 5.0) == 0.0
# Nonzero four-term contribution check (PGD case from review)
nonzero_expanded = epk.EvaluatedPathKernel.four_term_contribution(0, 1, 2.0, 3.0, 1, 5.0)
assert nonzero_expanded == 11.0

# Execute checklist item 3: Disjoint-support refusal test
try:
    epk.ReliabilityModels.normalized_product_of_experts([[1.0, 0.0], [0.0, 1.0]])
    assert False, "Expected ValueError on disjoint supports"
except ValueError as err:
    assert "positive normalization denominator" in str(err)
    print({'disjoint-support refusal': 'PASS', 'error': str(err)})

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(['signed mean', 'absolute defect rate'], [report.mean_delta_e, report.egd], color=['#e45756', '#72b7b2'])
ax.set_ylim(-0.1, 1.1)
ax.set_title('Cancellation hides pointwise disagreement')
plt.show()

## Toggle the missing path

The control inserts or removes the sole compatible concatenation. In a headless run it exercises the failing case.

In [ ]:
def probe_g1(include_global_path: bool) -> None:
    paths = [epk.ComposedTrajectory(left, right)] if include_global_path else []
    ok, messages = boundary.validate_unique_decomposition(
        paths, left_segments=[left], right_segments=[right]
    )
    print({'G1': ok, 'violation count': len(messages)})

if HAS_WIDGETS:
    import ipywidgets as widgets
    from IPython.display import display
    display(widgets.interactive(probe_g1, include_global_path=widgets.Checkbox(value=False, description='include global path')))
else:
    probe_g1(False)

## Reviewer checklist

1. Remove one compatible concatenation and require G1 to fail.
2. Pair an NGD with a PGD and require the signed residual to vanish while EGD remains positive.
3. Give normalized Product of Experts disjoint supports and require a positive-denominator error.
4. Compare the manuscript statements with `C1c_g1_set_isomorphism`, `D3_fully_corrected_kernel_identity`, `D4_cancellation_not_pointwise_zero`, `E2_count_gap_eq_delta_sum`, and `E6_egd_zero_iff_all_zdr`.

Passing these finite checks does not establish evaluator adequacy or deployment reliability.